# 1.1 — Imports

In [1]:
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# 1.2 — Root path and folder setup

In [ ]:
ROOT_DIR = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "Notebook").exists() and (path / "Data").exists()
    ),
    None,
)

if ROOT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from inside the cloned repository."
    )

RAW_DIR = ROOT_DIR / "Data" / "Raw"
PROCESSED_DIR = ROOT_DIR / "Data" / "Processed"
FEATURE_DIR = PROCESSED_DIR / "aggregated_features"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT_DIR      :", ROOT_DIR)
print("RAW_DIR       :", RAW_DIR)
print("PROCESSED_DIR :", PROCESSED_DIR)
print("FEATURE_DIR   :", FEATURE_DIR)


# 1.3 — Required file check

In [3]:
expected_files = [
    "application_train.csv",
    "application_test.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "installments_payments.csv",
    "POS_CASH_balance.csv",
    "credit_card_balance.csv",
    "sample_submission.csv",
]

file_check = pd.DataFrame({
    "file_name": expected_files,
    "exists": [(RAW_DIR / f).exists() for f in expected_files]
})

display(file_check)

assert file_check["exists"].all(), "Some required files are missing inside Data/Raw"

,file_name,exists
0,application_train.csv,True
1,application_test.csv,True
2,bureau.csv,True
3,bureau_balance.csv,True
4,previous_application.csv,True
5,installments_payments.csv,True
6,POS_CASH_balance.csv,True
7,credit_card_balance.csv,True
8,sample_submission.csv,True


# 2.1 — Load all CSV files

In [4]:
application_train = pd.read_csv(RAW_DIR / "application_train.csv", low_memory=False)
application_test = pd.read_csv(RAW_DIR / "application_test.csv", low_memory=False)

bureau = pd.read_csv(RAW_DIR / "bureau.csv", low_memory=False)
bureau_balance = pd.read_csv(RAW_DIR / "bureau_balance.csv", low_memory=False)
previous_application = pd.read_csv(RAW_DIR / "previous_application.csv", low_memory=False)
installments_payments = pd.read_csv(RAW_DIR / "installments_payments.csv", low_memory=False)
pos_cash_balance = pd.read_csv(RAW_DIR / "POS_CASH_balance.csv", low_memory=False)
credit_card_balance = pd.read_csv(RAW_DIR / "credit_card_balance.csv", low_memory=False)

sample_submission = pd.read_csv(RAW_DIR / "sample_submission.csv", low_memory=False)

# 2.2 — Quick sanity check

In [5]:
datasets = {
    "application_train": application_train,
    "application_test": application_test,
    "bureau": bureau,
    "bureau_balance": bureau_balance,
    "previous_application": previous_application,
    "installments_payments": installments_payments,
    "pos_cash_balance": pos_cash_balance,
    "credit_card_balance": credit_card_balance,
    "sample_submission": sample_submission,
}

shape_df = pd.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in datasets.items()],
    columns=["dataset", "rows", "cols"]
)

display(shape_df)

print("Train TARGET null count :", application_train["TARGET"].isna().sum())
print("Train duplicate SK_ID_CURR :", application_train["SK_ID_CURR"].duplicated().sum())
print("Test duplicate SK_ID_CURR  :", application_test["SK_ID_CURR"].duplicated().sum())

,dataset,rows,cols
0,application_train,307511,122
1,application_test,48744,121
2,bureau,1716428,17
3,bureau_balance,27299925,3
4,previous_application,1670214,37
5,installments_payments,13605401,8
6,pos_cash_balance,10001358,8
7,credit_card_balance,3840312,23
8,sample_submission,48744,2


Train TARGET null count : 0
Train duplicate SK_ID_CURR : 0
Test duplicate SK_ID_CURR  : 0


# 3.1 — Helper functions

In [6]:
def safe_divide(numerator, denominator):
    if isinstance(denominator, pd.Series):
        denominator = denominator.replace({0: np.nan})
    return numerator / denominator


def one_hot_encoder(df, nan_as_category=True):
    original_columns = list(df.columns)
    categorical_columns = [col for col in df.columns if df[col].dtype == "object"]

    if len(categorical_columns) == 0:
        return df.copy(), []

    df = pd.get_dummies(df, columns=categorical_columns, dummy_na=nan_as_category)
    new_columns = [c for c in df.columns if c not in original_columns]
    return df, new_columns


def agg_numeric(df, group_var, agg_dict, prefix):
    grouped = df.groupby(group_var).agg(agg_dict)
    grouped.columns = [f"{prefix}{col}_{stat.upper()}" for col, stat in grouped.columns]
    grouped = grouped.reset_index()
    return grouped


def agg_categorical(df, group_var, prefix, nan_as_category=True):
    categorical_columns = [col for col in df.columns if df[col].dtype == "object"]

    if len(categorical_columns) == 0:
        return pd.DataFrame({group_var: df[group_var].drop_duplicates().sort_values().values})

    cat_df = df[[group_var] + categorical_columns].copy()
    cat_df, _ = one_hot_encoder(cat_df, nan_as_category=nan_as_category)
    cat_agg = cat_df.groupby(group_var).mean().reset_index()

    rename_dict = {col: f"{prefix}{col}_MEAN" for col in cat_agg.columns if col != group_var}
    cat_agg = cat_agg.rename(columns=rename_dict)
    return cat_agg


def add_missing_flags(df):
    df = df.copy()
    df["APP_MISSING_COUNT"] = df.isnull().sum(axis=1)
    df["APP_MISSING_RATIO"] = df.isnull().mean(axis=1)
    return df

# 3.2 — Application base feature function

In [7]:
def build_application_features(df):
    df = df.copy()

    # Sentinel fix
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(np.int8)
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

    # Core ratios
    df["APP_CREDIT_INCOME_RATIO"] = safe_divide(df["AMT_CREDIT"], df["AMT_INCOME_TOTAL"])
    df["APP_ANNUITY_INCOME_RATIO"] = safe_divide(df["AMT_ANNUITY"], df["AMT_INCOME_TOTAL"])
    df["APP_CREDIT_ANNUITY_RATIO"] = safe_divide(df["AMT_CREDIT"], df["AMT_ANNUITY"])
    df["APP_GOODS_CREDIT_RATIO"] = safe_divide(df["AMT_GOODS_PRICE"], df["AMT_CREDIT"])
    df["APP_INCOME_PER_PERSON"] = safe_divide(df["AMT_INCOME_TOTAL"], df["CNT_FAM_MEMBERS"])
    df["APP_EMPLOYED_BIRTH_RATIO"] = safe_divide(df["DAYS_EMPLOYED"], df["DAYS_BIRTH"])
    df["APP_CAR_BIRTH_RATIO"] = safe_divide(df["OWN_CAR_AGE"], df["DAYS_BIRTH"])
    df["APP_PHONE_BIRTH_RATIO"] = safe_divide(df["DAYS_LAST_PHONE_CHANGE"], df["DAYS_BIRTH"])
    df["APP_PHONE_EMPLOYED_RATIO"] = safe_divide(df["DAYS_LAST_PHONE_CHANGE"], df["DAYS_EMPLOYED"])

    # EXT_SOURCE summary
    ext_cols = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in df.columns]
    df["APP_EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
    df["APP_EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)
    df["APP_EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
    df["APP_EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)

    # Document flags
    doc_cols = [c for c in df.columns if c.startswith("FLAG_DOCUMENT")]
    if len(doc_cols) > 0:
        df["APP_DOCUMENT_COUNT"] = df[doc_cols].sum(axis=1)

    # Contact flags
    contact_cols = [
        c for c in [
            "FLAG_MOBIL",
            "FLAG_EMP_PHONE",
            "FLAG_WORK_PHONE",
            "FLAG_CONT_MOBILE",
            "FLAG_PHONE",
            "FLAG_EMAIL",
        ] if c in df.columns
    ]
    if len(contact_cols) > 0:
        df["APP_CONTACT_COUNT"] = df[contact_cols].sum(axis=1)

    # Address mismatch style features
    addr_cols = [
        c for c in [
            "REG_REGION_NOT_LIVE_REGION",
            "REG_REGION_NOT_WORK_REGION",
            "LIVE_REGION_NOT_WORK_REGION",
            "REG_CITY_NOT_LIVE_CITY",
            "REG_CITY_NOT_WORK_CITY",
            "LIVE_CITY_NOT_WORK_CITY",
        ] if c in df.columns
    ]
    if len(addr_cols) > 0:
        df["APP_ADDRESS_MISMATCH_COUNT"] = df[addr_cols].sum(axis=1)

    # Missing summary
    df = add_missing_flags(df)

    return df

# 4.1 — Bureau balance aggregation function

In [8]:
def build_bureau_balance_features(bureau_balance):
    bb = bureau_balance.copy()

    bb["STATUS_BAD_FLAG"] = bb["STATUS"].isin(["1", "2", "3", "4", "5"]).astype(np.int8)
    bb["STATUS_ZERO_FLAG"] = (bb["STATUS"] == "0").astype(np.int8)
    bb["STATUS_C_FLAG"] = (bb["STATUS"] == "C").astype(np.int8)
    bb["STATUS_X_FLAG"] = (bb["STATUS"] == "X").astype(np.int8)

    bb_num = agg_numeric(
        bb,
        "SK_ID_BUREAU",
        {
            "MONTHS_BALANCE": ["min", "max", "size"],
            "STATUS_BAD_FLAG": ["mean", "sum"],
            "STATUS_ZERO_FLAG": ["mean"],
            "STATUS_C_FLAG": ["mean"],
            "STATUS_X_FLAG": ["mean"],
        },
        prefix="BB_",
    )

    bb_cat = agg_categorical(bb, "SK_ID_BUREAU", prefix="BB_")
    bb_agg = bb_num.merge(bb_cat, on="SK_ID_BUREAU", how="left")

    return bb_agg

# 4.2 — Bureau aggregation function

In [9]:
def build_bureau_features(bureau, bureau_balance):
    bb_agg = build_bureau_balance_features(bureau_balance)

    bur = bureau.copy()
    bur = bur.merge(bb_agg, on="SK_ID_BUREAU", how="left")

    bur["BUREAU_DEBT_CREDIT_RATIO_RAW"] = safe_divide(bur["AMT_CREDIT_SUM_DEBT"], bur["AMT_CREDIT_SUM"])
    bur["BUREAU_OVERDUE_DEBT_RATIO_RAW"] = safe_divide(bur["AMT_CREDIT_SUM_OVERDUE"], bur["AMT_CREDIT_SUM_DEBT"])

    bur_num_agg = {
        "SK_ID_BUREAU": ["nunique"],
        "DAYS_CREDIT": ["min", "max", "mean", "var"],
        "DAYS_CREDIT_ENDDATE": ["min", "max", "mean"],
        "DAYS_CREDIT_UPDATE": ["min", "max", "mean"],
        "CREDIT_DAY_OVERDUE": ["max", "mean", "sum"],
        "AMT_CREDIT_MAX_OVERDUE": ["max", "mean"],
        "AMT_CREDIT_SUM": ["max", "mean", "sum"],
        "AMT_CREDIT_SUM_DEBT": ["max", "mean", "sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["max", "mean", "sum"],
        "AMT_CREDIT_SUM_LIMIT": ["max", "mean", "sum"],
        "AMT_ANNUITY": ["max", "mean"],
        "CNT_CREDIT_PROLONG": ["sum", "mean"],
        "BUREAU_DEBT_CREDIT_RATIO_RAW": ["mean", "max"],
        "BUREAU_OVERDUE_DEBT_RATIO_RAW": ["mean", "max"],
    }

    bb_feature_cols = [c for c in bur.columns if c.startswith("BB_")]
    for c in bb_feature_cols:
        bur_num_agg[c] = ["mean"]

    bur_num = agg_numeric(bur, "SK_ID_CURR", bur_num_agg, prefix="BUREAU_")
    bur_cat = agg_categorical(bur, "SK_ID_CURR", prefix="BUREAU_")
    bureau_agg = bur_num.merge(bur_cat, on="SK_ID_CURR", how="left")

    active = bur[bur["CREDIT_ACTIVE"] == "Active"]
    closed = bur[bur["CREDIT_ACTIVE"] == "Closed"]

    if not active.empty:
        active_agg = agg_numeric(
            active,
            "SK_ID_CURR",
            {
                "AMT_CREDIT_SUM": ["sum", "mean"],
                "AMT_CREDIT_SUM_DEBT": ["sum", "mean"],
                "AMT_CREDIT_SUM_OVERDUE": ["sum", "mean"],
                "DAYS_CREDIT": ["min", "max", "mean"],
            },
            prefix="BUREAU_ACTIVE_",
        )
        bureau_agg = bureau_agg.merge(active_agg, on="SK_ID_CURR", how="left")

    if not closed.empty:
        closed_agg = agg_numeric(
            closed,
            "SK_ID_CURR",
            {
                "AMT_CREDIT_SUM": ["sum", "mean"],
                "DAYS_CREDIT": ["min", "max", "mean"],
            },
            prefix="BUREAU_CLOSED_",
        )
        bureau_agg = bureau_agg.merge(closed_agg, on="SK_ID_CURR", how="left")

    if (
        "BUREAU_AMT_CREDIT_SUM_DEBT_SUM" in bureau_agg.columns
        and "BUREAU_AMT_CREDIT_SUM_SUM" in bureau_agg.columns
    ):
        bureau_agg["BUREAU_TOTAL_DEBT_CREDIT_RATIO"] = safe_divide(
            bureau_agg["BUREAU_AMT_CREDIT_SUM_DEBT_SUM"],
            bureau_agg["BUREAU_AMT_CREDIT_SUM_SUM"]
        )

    if (
        "BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM" in bureau_agg.columns
        and "BUREAU_AMT_CREDIT_SUM_DEBT_SUM" in bureau_agg.columns
    ):
        bureau_agg["BUREAU_TOTAL_OVERDUE_DEBT_RATIO"] = safe_divide(
            bureau_agg["BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM"],
            bureau_agg["BUREAU_AMT_CREDIT_SUM_DEBT_SUM"]
        )

    return bureau_agg

# 5.1 — Previous application aggregation function

In [10]:
def build_previous_features(previous):
    prev = previous.copy()

    day_cols = [
        "DAYS_FIRST_DRAWING",
        "DAYS_FIRST_DUE",
        "DAYS_LAST_DUE_1ST_VERSION",
        "DAYS_LAST_DUE",
        "DAYS_TERMINATION",
    ]

    for col in day_cols:
        if col in prev.columns:
            prev[col] = prev[col].replace(365243, np.nan)

    prev["APP_CREDIT_PERC"] = safe_divide(prev["AMT_APPLICATION"], prev["AMT_CREDIT"])
    prev["CREDIT_TO_ANNUITY_RATIO"] = safe_divide(prev["AMT_CREDIT"], prev["AMT_ANNUITY"])
    prev["DOWN_PAYMENT_TO_CREDIT"] = safe_divide(prev["AMT_DOWN_PAYMENT"], prev["AMT_CREDIT"])
    prev["GOODS_TO_CREDIT_RATIO"] = safe_divide(prev["AMT_GOODS_PRICE"], prev["AMT_CREDIT"])

    prev_num = agg_numeric(
        prev,
        "SK_ID_CURR",
        {
            "SK_ID_PREV": ["nunique"],
            "AMT_ANNUITY": ["max", "mean"],
            "AMT_APPLICATION": ["max", "mean"],
            "AMT_CREDIT": ["max", "mean"],
            "APP_CREDIT_PERC": ["max", "mean", "var"],
            "AMT_DOWN_PAYMENT": ["max", "mean"],
            "AMT_GOODS_PRICE": ["max", "mean"],
            "HOUR_APPR_PROCESS_START": ["max", "mean"],
            "RATE_DOWN_PAYMENT": ["max", "mean"],
            "DAYS_DECISION": ["min", "max", "mean"],
            "CNT_PAYMENT": ["mean", "sum"],
            "CREDIT_TO_ANNUITY_RATIO": ["max", "mean"],
            "DOWN_PAYMENT_TO_CREDIT": ["max", "mean"],
            "GOODS_TO_CREDIT_RATIO": ["max", "mean"],
        },
        prefix="PREV_",
    )

    prev_cat = agg_categorical(prev, "SK_ID_CURR", prefix="PREV_")
    prev_agg = prev_num.merge(prev_cat, on="SK_ID_CURR", how="left")

    approved = prev[prev["NAME_CONTRACT_STATUS"] == "Approved"]
    refused = prev[prev["NAME_CONTRACT_STATUS"] == "Refused"]

    if not approved.empty:
        approved_agg = agg_numeric(
            approved,
            "SK_ID_CURR",
            {
                "SK_ID_PREV": ["nunique"],
                "AMT_APPLICATION": ["max", "mean"],
                "AMT_CREDIT": ["max", "mean"],
                "APP_CREDIT_PERC": ["max", "mean"],
                "DAYS_DECISION": ["min", "mean"],
                "CNT_PAYMENT": ["mean", "sum"],
            },
            prefix="APPROVED_",
        )
        prev_agg = prev_agg.merge(approved_agg, on="SK_ID_CURR", how="left")

    if not refused.empty:
        refused_agg = agg_numeric(
            refused,
            "SK_ID_CURR",
            {
                "SK_ID_PREV": ["nunique"],
                "AMT_APPLICATION": ["max", "mean"],
                "AMT_CREDIT": ["max", "mean"],
                "APP_CREDIT_PERC": ["max", "mean"],
                "DAYS_DECISION": ["min", "mean"],
            },
            prefix="REFUSED_",
        )
        prev_agg = prev_agg.merge(refused_agg, on="SK_ID_CURR", how="left")

    return prev_agg

# 6.1 — Installments aggregation function

In [11]:
def build_installments_features(installments):
    ins = installments.copy()

    ins["PAY_PERC"] = safe_divide(ins["AMT_PAYMENT"], ins["AMT_INSTALMENT"])
    ins["PAY_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]
    ins["DPD"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0)
    ins["DBD"] = (ins["DAYS_INSTALMENT"] - ins["DAYS_ENTRY_PAYMENT"]).clip(lower=0)

    ins["LATE_PAYMENT_FLAG"] = (ins["DPD"] > 0).astype(np.int8)
    ins["UNDERPAY_FLAG"] = (ins["AMT_PAYMENT"] < ins["AMT_INSTALMENT"]).astype(np.int8)

    ins_agg = agg_numeric(
        ins,
        "SK_ID_CURR",
        {
            "SK_ID_PREV": ["nunique"],
            "NUM_INSTALMENT_VERSION": ["nunique"],
            "DAYS_INSTALMENT": ["min", "max", "mean"],
            "DAYS_ENTRY_PAYMENT": ["min", "max", "mean"],
            "AMT_INSTALMENT": ["max", "mean", "sum"],
            "AMT_PAYMENT": ["min", "max", "mean", "sum"],
            "PAY_PERC": ["max", "mean", "var"],
            "PAY_DIFF": ["max", "mean", "sum", "var"],
            "DPD": ["max", "mean", "sum"],
            "DBD": ["max", "mean", "sum"],
            "LATE_PAYMENT_FLAG": ["mean", "sum"],
            "UNDERPAY_FLAG": ["mean", "sum"],
        },
        prefix="INS_",
    )

    ins_recent = ins[ins["DAYS_INSTALMENT"] >= -365]
    if not ins_recent.empty:
        ins_recent_agg = agg_numeric(
            ins_recent,
            "SK_ID_CURR",
            {
                "AMT_PAYMENT": ["mean", "sum"],
                "PAY_PERC": ["mean", "max"],
                "PAY_DIFF": ["mean", "sum"],
                "DPD": ["mean", "max", "sum"],
                "LATE_PAYMENT_FLAG": ["mean", "sum"],
            },
            prefix="INS_LAST365_",
        )
        ins_agg = ins_agg.merge(ins_recent_agg, on="SK_ID_CURR", how="left")

    return ins_agg

# 7.1 — POS_CASH aggregation function

In [12]:
def build_pos_cash_features(pos_cash):
    pos = pos_cash.copy()

    pos["POS_LATE_FLAG"] = (pos["SK_DPD"] > 0).astype(np.int8)

    pos_num = agg_numeric(
        pos,
        "SK_ID_CURR",
        {
            "SK_ID_PREV": ["nunique"],
            "MONTHS_BALANCE": ["min", "max", "size"],
            "CNT_INSTALMENT": ["max", "mean", "sum"],
            "CNT_INSTALMENT_FUTURE": ["max", "mean", "sum"],
            "SK_DPD": ["max", "mean", "sum"],
            "SK_DPD_DEF": ["max", "mean", "sum"],
            "POS_LATE_FLAG": ["mean", "sum"],
        },
        prefix="POS_",
    )

    pos_cat = agg_categorical(pos, "SK_ID_CURR", prefix="POS_")
    pos_agg = pos_num.merge(pos_cat, on="SK_ID_CURR", how="left")

    return pos_agg

# 8.1 — Credit card aggregation function

In [13]:
def build_credit_card_features(credit_card):
    cc = credit_card.copy()

    cc["BALANCE_LIMIT_RATIO"] = safe_divide(cc["AMT_BALANCE"], cc["AMT_CREDIT_LIMIT_ACTUAL"])
    cc["DRAWING_LIMIT_RATIO"] = safe_divide(cc["AMT_DRAWINGS_CURRENT"], cc["AMT_CREDIT_LIMIT_ACTUAL"])
    cc["PAYMENT_MIN_RATIO"] = safe_divide(cc["AMT_PAYMENT_CURRENT"], cc["AMT_INST_MIN_REGULARITY"])
    cc["PAYMENT_TOTAL_RATIO"] = safe_divide(cc["AMT_PAYMENT_TOTAL_CURRENT"], cc["AMT_TOTAL_RECEIVABLE"])
    cc["RECEIVABLE_LIMIT_RATIO"] = safe_divide(cc["AMT_TOTAL_RECEIVABLE"], cc["AMT_CREDIT_LIMIT_ACTUAL"])
    cc["CC_LATE_FLAG"] = (cc["SK_DPD"] > 0).astype(np.int8)

    cc_num = agg_numeric(
        cc,
        "SK_ID_CURR",
        {
            "SK_ID_PREV": ["nunique"],
            "MONTHS_BALANCE": ["min", "max", "size"],
            "AMT_BALANCE": ["max", "mean", "sum"],
            "AMT_CREDIT_LIMIT_ACTUAL": ["max", "mean"],
            "AMT_DRAWINGS_ATM_CURRENT": ["max", "mean", "sum"],
            "AMT_DRAWINGS_CURRENT": ["max", "mean", "sum"],
            "AMT_DRAWINGS_POS_CURRENT": ["max", "mean", "sum"],
            "AMT_INST_MIN_REGULARITY": ["max", "mean"],
            "AMT_PAYMENT_CURRENT": ["max", "mean", "sum"],
            "AMT_PAYMENT_TOTAL_CURRENT": ["max", "mean", "sum"],
            "AMT_RECEIVABLE_PRINCIPAL": ["max", "mean", "sum"],
            "AMT_TOTAL_RECEIVABLE": ["max", "mean", "sum"],
            "CNT_DRAWINGS_ATM_CURRENT": ["max", "mean", "sum"],
            "CNT_DRAWINGS_CURRENT": ["max", "mean", "sum"],
            "CNT_DRAWINGS_POS_CURRENT": ["max", "mean", "sum"],
            "CNT_INSTALMENT_MATURE_CUM": ["max", "mean"],
            "SK_DPD": ["max", "mean", "sum"],
            "SK_DPD_DEF": ["max", "mean", "sum"],
            "BALANCE_LIMIT_RATIO": ["max", "mean"],
            "DRAWING_LIMIT_RATIO": ["max", "mean"],
            "PAYMENT_MIN_RATIO": ["max", "mean"],
            "PAYMENT_TOTAL_RATIO": ["max", "mean"],
            "RECEIVABLE_LIMIT_RATIO": ["max", "mean"],
            "CC_LATE_FLAG": ["mean", "sum"],
        },
        prefix="CC_",
    )

    cc_cat = agg_categorical(cc, "SK_ID_CURR", prefix="CC_")
    cc_agg = cc_num.merge(cc_cat, on="SK_ID_CURR", how="left")

    return cc_agg

# 9.1 — Build all aggregated feature tables

In [14]:
bureau_agg = build_bureau_features(bureau, bureau_balance)
previous_agg = build_previous_features(previous_application)
installments_agg = build_installments_features(installments_payments)
pos_agg = build_pos_cash_features(pos_cash_balance)
cc_agg = build_credit_card_features(credit_card_balance)

agg_tables = {
    "bureau_agg": bureau_agg,
    "previous_agg": previous_agg,
    "installments_agg": installments_agg,
    "pos_agg": pos_agg,
    "cc_agg": cc_agg,
}

for name, df in agg_tables.items():
    print(f"{name:20s} -> shape: {df.shape} | duplicate SK_ID_CURR: {df['SK_ID_CURR'].duplicated().sum()}")

bureau_agg           -> shape: (305811, 96) | duplicate SK_ID_CURR: 0
previous_agg         -> shape: (338857, 209) | duplicate SK_ID_CURR: 0
installments_agg     -> shape: (339587, 44) | duplicate SK_ID_CURR: 0
pos_agg              -> shape: (337252, 29) | duplicate SK_ID_CURR: 0
cc_agg               -> shape: (103558, 70) | duplicate SK_ID_CURR: 0


## 9.2 — Build base train/test and merge all features

In [15]:
train_base = build_application_features(application_train.copy())
test_base = build_application_features(application_test.copy())

feature_frames = [
    bureau_agg,
    previous_agg,
    installments_agg,
    pos_agg,
    cc_agg,
]

for feat_df in feature_frames:
    train_base = train_base.merge(feat_df, on="SK_ID_CURR", how="left")
    test_base = test_base.merge(feat_df, on="SK_ID_CURR", how="left")

final_train = train_base.copy()
final_test = test_base.copy()

print("final_train shape:", final_train.shape)
print("final_test shape :", final_test.shape)

final_train shape: (307511, 584)
final_test shape : (48744, 583)


# 9.3 — Align train/test columns and sanity check

In [16]:
train_cols_wo_target = [c for c in final_train.columns if c != "TARGET"]
test_cols = list(final_test.columns)

missing_in_test = sorted(set(train_cols_wo_target) - set(test_cols))
missing_in_train = sorted(set(test_cols) - set(train_cols_wo_target))

for col in missing_in_test:
    final_test[col] = np.nan

for col in missing_in_train:
    final_train[col] = np.nan

feature_cols = [c for c in final_train.columns if c not in ["TARGET"]]

final_test = final_test[feature_cols]
final_train = final_train[["SK_ID_CURR", "TARGET"] + [c for c in feature_cols if c != "SK_ID_CURR"]]

print("final_train shape:", final_train.shape)
print("final_test shape :", final_test.shape)
print("Train duplicate SK_ID_CURR:", final_train["SK_ID_CURR"].duplicated().sum())
print("Test duplicate SK_ID_CURR :", final_test["SK_ID_CURR"].duplicated().sum())
print("Train TARGET null count   :", final_train["TARGET"].isna().sum())

feature_diff = sorted((set(final_train.columns) - {"TARGET"}) ^ set(final_test.columns))
print("Column difference except TARGET:", len(feature_diff))

final_train shape: (307511, 584)
final_test shape : (48744, 583)
Train duplicate SK_ID_CURR: 0
Test duplicate SK_ID_CURR : 0
Train TARGET null count   : 0
Column difference except TARGET: 0


# 9.4 — Optional: save intermediate aggregated tables

In [ ]:
for name, df in agg_tables.items():
    csv_path = FEATURE_DIR / f"{name}.csv"
    df.to_csv(csv_path, index=False)

print("Intermediate aggregated tables saved in:", FEATURE_DIR)

# 10.1 — Save final train/test files

In [ ]:
train_csv_path = PROCESSED_DIR / "final_train_before_eda.csv"
test_csv_path = PROCESSED_DIR / "final_test_before_eda.csv"

final_train.to_csv(train_csv_path, index=False)
final_test.to_csv(test_csv_path, index=False)

print("CSV saved successfully:")
print(train_csv_path)
print(test_csv_path)

try:
    train_parquet_path = PROCESSED_DIR / "final_train_before_eda.parquet"
    test_parquet_path = PROCESSED_DIR / "final_test_before_eda.parquet"

    final_train.to_parquet(train_parquet_path, index=False)
    final_test.to_parquet(test_parquet_path, index=False)

    print("\nParquet saved successfully:")
    print(train_parquet_path)
    print(test_parquet_path)

except Exception as e:
    print("\nParquet save skipped:", e)

# 10.2 — Final preview

In [19]:
print("TRAIN")
display(final_train.head())

print("TEST")
display(final_test.head())

TRAIN


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,DAYS_EMPLOYED_ANOM,APP_CREDIT_INCOME_RATIO,APP_ANNUITY_INCOME_RATIO,APP_CREDIT_ANNUITY_RATIO,APP_GOODS_CREDIT_RATIO,APP_INCOME_PER_PERSON,APP_EMPLOYED_BIRTH_RATIO,APP_CAR_BIRTH_RATIO,APP_PHONE_BIRTH_RATIO,APP_PHONE_EMPLOYED_RATIO,APP_EXT_SOURCE_MEAN,APP_EXT_SOURCE_MAX,APP_EXT_SOURCE_MIN,APP_EXT_SOURCE_STD,APP_DOCUMENT_COUNT,APP_CONTACT_COUNT,APP_ADDRESS_MISMATCH_COUNT,APP_MISSING_COUNT,APP_MISSING_RATIO,BUREAU_SK_ID_BUREAU_NUNIQUE,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_VAR,BUREAU_DAYS_CREDIT_ENDDATE_MIN,BUREAU_DAYS_CREDIT_ENDDATE_MAX,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MIN,...,APPROVED_CNT_PAYMENT_SUM,REFUSED_SK_ID_PREV_NUNIQUE,REFUSED_AMT_APPLICATION_MAX,REFUSED_AMT_APPLICATION_MEAN,REFUSED_AMT_CREDIT_MAX,REFUSED_AMT_CREDIT_MEAN,REFUSED_APP_CREDIT_PERC_MAX,REFUSED_APP_CREDIT_PERC_MEAN,REFUSED_DAYS_DECISION_MIN,REFUSED_DAYS_DECISION_MEAN,INS_SK_ID_PREV_NUNIQUE,INS_NUM_INSTALMENT_VERSION_NUNIQUE,INS_DAYS_INSTALMENT_MIN,INS_DAYS_INSTALMENT_MAX,INS_DAYS_INSTALMENT_MEAN,INS_DAYS_ENTRY_PAYMENT_MIN,INS_DAYS_ENTRY_PAYMENT_MAX,INS_DAYS_ENTRY_PAYMENT_MEAN,INS_AMT_INSTALMENT_MAX,INS_AMT_INSTALMENT_MEAN,INS_AMT_INSTALMENT_SUM,INS_AMT_PAYMENT_MIN,INS_AMT_PAYMENT_MAX,INS_AMT_PAYMENT_MEAN,INS_AMT_PAYMENT_SUM,INS_PAY_PERC_MAX,INS_PAY_PERC_MEAN,INS_PAY_PERC_VAR,INS_PAY_DIFF_MAX,INS_PAY_DIFF_MEAN,INS_PAY_DIFF_SUM,INS_PAY_DIFF_VAR,INS_DPD_MAX,INS_DPD_MEAN,INS_DPD_SUM,INS_DBD_MAX,INS_DBD_MEAN,INS_DBD_SUM,INS_LATE_PAYMENT_FLAG_MEAN,INS_LATE_PAYMENT_FLAG_SUM,INS_UNDERPAY_FLAG_MEAN,INS_UNDERPAY_FLAG_SUM,INS_LAST365_AMT_PAYMENT_MEAN,INS_LAST365_AMT_PAYMENT_SUM,INS_LAST365_PAY_PERC_MEAN,INS_LAST365_PAY_PERC_MAX,INS_LAST365_PAY_DIFF_MEAN,INS_LAST365_PAY_DIFF_SUM,INS_LAST365_DPD_MEAN,INS_LAST365_DPD_MAX,INS_LAST365_DPD_SUM

TEST


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,DAYS_EMPLOYED_ANOM,APP_CREDIT_INCOME_RATIO,APP_ANNUITY_INCOME_RATIO,APP_CREDIT_ANNUITY_RATIO,APP_GOODS_CREDIT_RATIO,APP_INCOME_PER_PERSON,APP_EMPLOYED_BIRTH_RATIO,APP_CAR_BIRTH_RATIO,APP_PHONE_BIRTH_RATIO,APP_PHONE_EMPLOYED_RATIO,APP_EXT_SOURCE_MEAN,APP_EXT_SOURCE_MAX,APP_EXT_SOURCE_MIN,APP_EXT_SOURCE_STD,APP_DOCUMENT_COUNT,APP_CONTACT_COUNT,APP_ADDRESS_MISMATCH_COUNT,APP_MISSING_COUNT,APP_MISSING_RATIO,BUREAU_SK_ID_BUREAU_NUNIQUE,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_VAR,BUREAU_DAYS_CREDIT_ENDDATE_MIN,BUREAU_DAYS_CREDIT_ENDDATE_MAX,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MIN,BUREAU_DAYS_CREDIT_UPDATE_MAX,...,APPROVED_CNT_PAYMENT_SUM,REFUSED_SK_ID_PREV_NUNIQUE,REFUSED_AMT_APPLICATION_MAX,REFUSED_AMT_APPLICATION_MEAN,REFUSED_AMT_CREDIT_MAX,REFUSED_AMT_CREDIT_MEAN,REFUSED_APP_CREDIT_PERC_MAX,REFUSED_APP_CREDIT_PERC_MEAN,REFUSED_DAYS_DECISION_MIN,REFUSED_DAYS_DECISION_MEAN,INS_SK_ID_PREV_NUNIQUE,INS_NUM_INSTALMENT_VERSION_NUNIQUE,INS_DAYS_INSTALMENT_MIN,INS_DAYS_INSTALMENT_MAX,INS_DAYS_INSTALMENT_MEAN,INS_DAYS_ENTRY_PAYMENT_MIN,INS_DAYS_ENTRY_PAYMENT_MAX,INS_DAYS_ENTRY_PAYMENT_MEAN,INS_AMT_INSTALMENT_MAX,INS_AMT_INSTALMENT_MEAN,INS_AMT_INSTALMENT_SUM,INS_AMT_PAYMENT_MIN,INS_AMT_PAYMENT_MAX,INS_AMT_PAYMENT_MEAN,INS_AMT_PAYMENT_SUM,INS_PAY_PERC_MAX,INS_PAY_PERC_MEAN,INS_PAY_PERC_VAR,INS_PAY_DIFF_MAX,INS_PAY_DIFF_MEAN,INS_PAY_DIFF_SUM,INS_PAY_DIFF_VAR,INS_DPD_MAX,INS_DPD_MEAN,INS_DPD_SUM,INS_DBD_MAX,INS_DBD_MEAN,INS_DBD_SUM,INS_LATE_PAYMENT_FLAG_MEAN,INS_LATE_PAYMENT_FLAG_SUM,INS_UNDERPAY_FLAG_MEAN,INS_UNDERPAY_FLAG_SUM,INS_LAST365_AMT_PAYMENT_MEAN,INS_LAST365_AMT_PAYMENT_SUM,INS_LAST365_PAY_PERC_MEAN,INS_LAST365_PAY_PERC_MAX,INS_LAST365_PAY_DIFF_MEAN,INS_LAST365_PAY_DIFF_SUM,INS_LAST365_DPD_MEAN,INS_LAST365_DPD_

# 10.3 — Final memory cleanup

In [20]:
gc.collect()
print("Done.")

Done.
